# 🛡️ DataOps Sentinel — Interactive Data Profiling, SQL Reconciliation & RCA Walkthrough
**Author:** Ankur Bhatt | Senior Analyst (Data Operations & Analytics)  
**Skills Demonstrated:** `Python`, `Pandas`, `NumPy`, `SQL`, `ETL Data Cleansing`, `Schema & Constraint Validation`, `ServiceNow/JIRA Root Cause Analysis`

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
from src.data_generator import generate_o2c_datasets
from src.etl_validator import profile_dataframe, run_o2c_etl_validation
from src.sql_workbench import build_sqlite_warehouse, execute_sql_query, PREBUILT_SQL_QUERIES
from src.incident_rca import build_incident_backlog, get_incident_rca_deep_dive

## 1. Ingest Multi-Source Order-to-Cash (O2C) Staging Feeds

In [ ]:
datasets = generate_o2c_datasets(n_orders=1200, scenario='P1 Incident: Payment Webhook Retry Storm')
for name, df in datasets.items():
    print(f'{name:20s}: {len(df):,d} rows x {len(df.columns)} cols')

## 2. Automated Column-Level Data Profiling (`orders_raw`)

In [ ]:
profile_dataframe(datasets['orders_raw'])

## 3. Execute 10-Rule Declarative Schema & Constraint Validation Gate

In [ ]:
val_output = run_o2c_etl_validation(datasets)
val_output['rule_results']

## 4. Run SQL Ledger Reconciliation in SQLite Operational Warehouse

In [ ]:
conn = build_sqlite_warehouse(datasets, val_output)
recon_sql = PREBUILT_SQL_QUERIES['1. Order-to-Payment Ledger Reconciliation (Variance > $0.05)']['sql']
recon_df, _ = execute_sql_query(conn, recon_sql)
recon_df.head(10)

## 5. ServiceNow / JIRA Incident Queue & 5-Whys Root Cause Analysis

In [ ]:
incidents_df = build_incident_backlog(val_output)
incidents_df